# 001 · Dates, Times and Timedeltas

> **Standard Library › Dates & Times** · Python 3.14

The `datetime` module gives you value types for calendar dates, wall-clock times, combined timestamps and durations — the same job `DateOnly`, `TimeOnly`, `DateTime` and `TimeSpan` do in .NET.

**In this notebook you will learn:**

- the four core types: `date`, `time`, `datetime` and `timedelta`, and their .NET counterparts
- how `timedelta` normalises its fields and how to get totals out of it
- date arithmetic, comparison and `replace()` (Python's `with`-style copy)
- weekday numbering differences (`weekday()`, `isoweekday()` vs `DayOfWeek`)
- why there is no `AddMonths`, and how to write one yourself

## .NET mental model

| .NET | Python (`datetime` module) | Notes |
|---|---|---|
| `DateOnly` | `date` | year, month, day |
| `TimeOnly` | `time` | hour … microsecond, optional `tzinfo` |
| `DateTime` / `DateTimeOffset` | `datetime` | a `datetime` *is a* `date` subclass |
| `TimeSpan` | `timedelta` | stored as days + seconds + microseconds |
| `DayOfWeek` enum | `weekday()` / `isoweekday()` ints | different numbering! |
| `DateTime.AddMonths` | — | not in the stdlib; write it, or use `dateutil` |

Like their .NET counterparts, all four types are **immutable** and **hashable**, so they are safe as dictionary keys. Resolution is **microseconds** (.NET ticks are 100 ns). Time zones are covered in *(see Standard Library › Dates & Times › 003 · Time Zones with zoneinfo)*; this notebook uses *naive* values (no time zone attached).

## Creating dates, times and datetimes

```csharp
// C#
var d  = new DateOnly(2026, 3, 5);
var t  = new TimeOnly(14, 30, 15);
var dt = new DateTime(2026, 3, 5, 14, 30, 15);
var combined = d.ToDateTime(t);
```

The usual import style is `from datetime import date, time, datetime, timedelta` — note the module and the class are both called `datetime`, which is a frequent source of confusion.

In [ ]:
from datetime import date, time, datetime, timedelta

d = date(2026, 3, 5)
t = time(14, 30, 15)
dt = datetime(2026, 3, 5, 14, 30, 15)
combined = datetime.combine(d, t)          # C#: d.ToDateTime(t)

print(d, t, dt, sep=" | ")
print(repr(dt))
print(combined == dt)                       # True

Every field is available as a read-only attribute, and you can split a `datetime` back into its parts:

In [ ]:
print(dt.year, dt.month, dt.day, dt.hour, dt.minute, dt.second, dt.microsecond)
print(dt.date(), dt.time())                 # C#: DateOnly.FromDateTime(dt), TimeOnly.FromDateTime(dt)

Invalid values fail immediately with `ValueError` (C#: `ArgumentOutOfRangeException`):

In [ ]:
try:
    date(2026, 2, 30)
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

## The current date and time

`date.today()` and `datetime.now()` are the equivalents of `DateTime.Today` and `DateTime.Now`. Their values change every run, so we only look at their types here. For UTC use `datetime.now(timezone.utc)` — covered in the time-zones notebook.

In [ ]:
today = date.today()            # C#: DateOnly.FromDateTime(DateTime.Today)
now = datetime.now()            # C#: DateTime.Now  (naive local time)

print(type(today).__name__, type(now).__name__)
print(now.date() == today)      # True (unless you run this exactly at midnight!)

**Tip:** code that calls `now()` deep inside business logic is hard to test. Pass the current time in as a parameter (or inject a clock function), just as you would inject `TimeProvider` in modern .NET.

## Ranges: `min`, `max` and `resolution`

| .NET | Python |
|---|---|
| `DateTime.MinValue` / `MaxValue` | `datetime.min` / `datetime.max` |
| `DateOnly.MinValue` / `MaxValue` | `date.min` / `date.max` |
| `TimeSpan.MaxValue` | `timedelta.max` |
| one tick (100 ns) | `timedelta.resolution` (1 µs) |

In [ ]:
print(date.min, "→", date.max)
print(datetime.max)
print(timedelta.max)
print(timedelta.resolution)

## `timedelta` — Python's `TimeSpan`

You construct a `timedelta` from any mix of keyword arguments (`weeks`, `days`, `hours`, `minutes`, `seconds`, `milliseconds`, `microseconds`). Internally it **normalises** everything into just three fields: `days`, `seconds` (0–86399) and `microseconds` (0–999999).

```csharp
// C#
var span = new TimeSpan(days: 1, hours: 36, minutes: 90, seconds: 0);
Console.WriteLine(span.Hours);        // component: 13
Console.WriteLine(span.TotalHours);   // 61.5
```

In [ ]:
span = timedelta(days=1, hours=36, minutes=90)

print(span)                    # 2 days, 13:30:00
print(repr(span))              # only days / seconds / microseconds are stored
print(span.days, span.seconds, span.microseconds)
print(span.total_seconds())    # 221400.0  (C#: span.TotalSeconds)

⚠️ **Gotcha:** there are no `.hours` or `.minutes` attributes, and `.seconds` is **not** the total — it is only the seconds *within the last day*. Use `total_seconds()` (or divide by another `timedelta`) for totals:

In [ ]:
print(span.seconds)                       # 48600 — NOT the total!
print(span.total_seconds() / 3600)        # 61.5 total hours (C#: TotalHours)
print(span / timedelta(hours=1))          # 61.5 — dividing two timedeltas gives a float
print(span // timedelta(hours=1))         # 61   — floor division gives an int

hours, remainder = divmod(span, timedelta(hours=1))
print(hours, remainder)                   # 61 0:30:00

⚠️ **Gotcha:** negative durations are normalised so that only `days` is negative. The string form looks odd but is correct:

In [ ]:
minus_one_hour = timedelta(hours=-1)
print(minus_one_hour)          # -1 day, 23:00:00
print(repr(minus_one_hour))    # days=-1, seconds=82800
print(minus_one_hour.total_seconds())   # -3600.0
print(abs(minus_one_hour))     # 1:00:00 (C#: span.Duration())

## Arithmetic

The operators work just like in C#: date ± timedelta → date, date − date → timedelta, and timedeltas can be added, multiplied and divided.

```csharp
// C#
var due = new DateOnly(2026, 3, 5).AddDays(30);
var gap = new DateTime(2026, 12, 25) - new DateTime(2026, 3, 5, 14, 30, 0);
```

In [ ]:
start = date(2026, 3, 5)
print(start + timedelta(days=30))          # 2026-04-04 (C#: AddDays(30))
print(start - timedelta(weeks=2))          # 2026-02-19

gap = datetime(2026, 12, 25) - datetime(2026, 3, 5, 14, 30)
print(gap)                                 # 294 days, 9:30:00
print(gap.days)                            # 294

print(timedelta(minutes=45) * 3)           # 2:15:00
print(timedelta(hours=1) / 4)              # 0:15:00

⚠️ **Gotcha:** a bare `time` does **not** support arithmetic (unlike `TimeOnly.Add`, which wraps around midnight). Combine it with a date first:

In [ ]:
try:
    time(23, 30) + timedelta(hours=1)
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

later = datetime.combine(date(2026, 3, 5), time(23, 30)) + timedelta(hours=1)
print(later)            # 2026-03-06 00:30:00
print(later.time())     # 00:30:00

## Comparison and sorting

Comparisons work with the normal operators, so `sorted`, `min` and `max` just work:

In [ ]:
deadlines = [date(2026, 6, 1), date(2026, 1, 15), date(2026, 3, 5)]
print(sorted(deadlines))
print(min(deadlines), max(deadlines))
print(date(2026, 3, 5) < date(2026, 3, 6))      # True

⚠️ **Gotcha:** `datetime` is a subclass of `date`, but they do not compare with each other: `==` is always `False` and `<` raises `TypeError`. Convert explicitly with `.date()` or `datetime.combine`:

In [ ]:
print(date(2026, 3, 5) == datetime(2026, 3, 5))          # False — different types
try:
    date(2026, 3, 5) < datetime(2026, 3, 5)
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

print(date(2026, 3, 5) == datetime(2026, 3, 5).date())   # True

## `replace()` — copy with changes

Because the types are immutable, you "modify" one by creating a new value with some fields replaced. It is the same idea as a C# `with` expression on a record.

```csharp
// C# — DateTime has no 'with', you rebuild it
var firstOfMonth = new DateTime(dt.Year, dt.Month, 1, dt.Hour, dt.Minute, dt.Second);
```

In [ ]:
dt = datetime(2026, 3, 5, 14, 30, 15)

print(dt.replace(day=1))                                  # first of the month
print(dt.replace(hour=0, minute=0, second=0))             # C#: dt.Date (still a datetime)
print(dt.replace(year=2027))
print(dt)                                                 # original is unchanged

**Tip:** since Python 3.13, the generic `copy.replace(obj, **changes)` also works on dates, times and datetimes — the same function you can use with dataclasses and named tuples: `copy.replace(date(2026, 3, 5), month=12)` gives `2026-12-05`.

## Day of the week

The three numbering schemes are all different, so be careful when porting code:

| Day | .NET `DayOfWeek` | Python `weekday()` | Python `isoweekday()` |
|---|---|---|---|
| Monday | 1 | **0** | 1 |
| Saturday | 6 | 5 | 6 |
| Sunday | **0** | 6 | **7** |

`isocalendar()` returns the ISO year, week number and weekday — like `System.Globalization.ISOWeek.GetWeekOfYear`.

In [ ]:
d = date(2026, 3, 5)          # a Thursday

print(d.weekday())            # 3   (Monday == 0)
print(d.isoweekday())         # 4   (Monday == 1)
print(d.strftime("%A"))       # Thursday (locale-dependent, see the formatting notebook)
print(d.isocalendar())        # ISO year, week and weekday
print(d.isocalendar().week)   # 10

Common use: find the next Monday, or check for weekends.

In [ ]:
def next_weekday(d: date, target: int) -> date:
    """Next date strictly after d whose weekday() == target (0 = Monday)."""
    days_ahead = (target - d.weekday() - 1) % 7 + 1
    return d + timedelta(days=days_ahead)

print(next_weekday(date(2026, 3, 5), 0))        # 2026-03-09 — the following Monday
print(date(2026, 3, 7).weekday() >= 5)          # True — Saturday is a weekend

## Adding months: there is no `AddMonths`

`timedelta` has no `months` or `years` argument, because their length varies. .NET's `AddMonths` clamps to the end of the month (`Jan 31 + 1 month = Feb 28`); in the stdlib you write that logic yourself, using `calendar.monthrange` to find the month length *(see Standard Library › Dates & Times › 005 · The calendar Module)*.

In [ ]:
import calendar

def add_months(d: date, months: int) -> date:
    """Like DateTime.AddMonths: clamps the day to the target month's length."""
    month_index = d.month - 1 + months
    year = d.year + month_index // 12
    month = month_index % 12 + 1
    last_day = calendar.monthrange(year, month)[1]
    return d.replace(year=year, month=month, day=min(d.day, last_day))

print(add_months(date(2026, 1, 31), 1))     # 2026-02-28 (clamped)
print(add_months(date(2028, 1, 31), 1))     # 2028-02-29 (leap year)
print(add_months(date(2026, 11, 15), 3))    # 2027-02-15 (crosses a year)
print(add_months(date(2026, 3, 31), -1))    # 2026-02-28 (going backwards)

⚠️ **Gotcha:** `d.replace(year=d.year + 1)` fails on 29 February. Handle it explicitly (or reuse `add_months(d, 12)`):

In [ ]:
leap_day = date(2028, 2, 29)
try:
    leap_day.replace(year=2029)
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

print(add_months(leap_day, 12))     # 2029-02-28

**Tip:** outside the standard library, the widely used third-party package `python-dateutil` provides `relativedelta(months=+1)` with exactly these semantics, plus a lot more (recurrence rules, fuzzy parsing). This course sticks to the stdlib.

## Ordinals: dates as day numbers

`toordinal()` converts a date to a day count (1 = 0001-01-01), just like `DateOnly.DayNumber` (which starts at 0). Handy for date maths and for iterating over ranges:

In [ ]:
print(date(2026, 3, 5).toordinal())
print(date.fromordinal(date(2026, 3, 5).toordinal() + 1))   # 2026-03-06

# all dates in a range (inclusive), like Enumerable.Range over DayNumber
first, last = date(2026, 2, 26), date(2026, 3, 2)
days = [first + timedelta(days=i) for i in range((last - first).days + 1)]
print([str(x) for x in days])

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Date only | `new DateOnly(2026, 3, 5)` | `date(2026, 3, 5)` |
| Time only | `new TimeOnly(14, 30)` | `time(14, 30)` |
| Date and time | `new DateTime(...)` | `datetime(...)` |
| Combine | `d.ToDateTime(t)` | `datetime.combine(d, t)` |
| Today / now | `DateTime.Today` / `DateTime.Now` | `date.today()` / `datetime.now()` |
| Duration | `TimeSpan.FromHours(2)` | `timedelta(hours=2)` |
| Total seconds | `span.TotalSeconds` | `span.total_seconds()` |
| Total hours | `span.TotalHours` | `span / timedelta(hours=1)` |
| Add days | `d.AddDays(30)` | `d + timedelta(days=30)` |
| Add months | `d.AddMonths(1)` | write a helper (or `dateutil.relativedelta`) |
| Copy with changes | rebuild / `with` on records | `dt.replace(day=1)` |
| Day of week | `d.DayOfWeek` (Sunday = 0) | `d.weekday()` (Monday = 0) / `d.isoweekday()` (Sunday = 7) |
| ISO week | `ISOWeek.GetWeekOfYear(dt)` | `d.isocalendar().week` |
| Range limits | `DateTime.MinValue` / `MaxValue` | `datetime.min` / `datetime.max` |

## Exercises

1. Write `age_on(birthday: date, on: date) -> int` that returns a person's age in whole years on a given day. Test with `date(2000, 2, 29)` on `date(2026, 2, 28)` (expect 25) and `date(2026, 3, 1)` (expect 26).
2. Given `timedelta(seconds=200_000)`, print it as `"2d 07h 33m 20s"` using `divmod`.
3. Write `business_days(start: date, end: date) -> int` that counts Monday–Friday dates in the half-open range `[start, end)`. Check that `date(2026, 3, 2)` to `date(2026, 3, 16)` gives 10.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
from datetime import date, timedelta

# 1 — compare (month, day) tuples to see whether the birthday has happened yet
def age_on(birthday: date, on: date) -> int:
    had_birthday = (on.month, on.day) >= (birthday.month, birthday.day)
    return on.year - birthday.year - (0 if had_birthday else 1)

print(age_on(date(2000, 2, 29), date(2026, 2, 28)))   # 25
print(age_on(date(2000, 2, 29), date(2026, 3, 1)))    # 26

# 2 — peel off days, hours and minutes with divmod
span = timedelta(seconds=200_000)
days, rest = divmod(span, timedelta(days=1))
hours, rest = divmod(rest, timedelta(hours=1))
minutes, rest = divmod(rest, timedelta(minutes=1))
print(f"{days}d {hours:02}h {minutes:02}m {rest.seconds:02}s")   # 2d 07h 33m 20s

# 3 — weekday() < 5 means Monday..Friday
def business_days(start: date, end: date) -> int:
    return sum(
        1
        for i in range((end - start).days)
        if (start + timedelta(days=i)).weekday() < 5
    )

print(business_days(date(2026, 3, 2), date(2026, 3, 16)))   # 10
```

</details>